#  Urban Data Intelligence – Valencia

**Objetivo:** construir un Índice de Calidad Urbana (UQI) por barrio combinando datos abiertos del Ayuntamiento de Valencia.

**Fuente de datos:** [opendata.vlci.valencia.es](https://opendata.vlci.valencia.es)

**Periodo de análisis:** 2016 – 2021

---

## Estructura del notebook

1. [Calidad del aire](#1-calidad-del-aire)
2. [Quejas ciudadanas](#2-quejas-ciudadanas)
3. [Ruido urbano](#3-ruido-urbano)
4. [Equipamientos municipales](#4-equipamientos-municipales)
5. [Valenbisi (movilidad)](#5-valenbisi-movilidad)
6. [Tabla maestra y cálculo del UQI](#6-tabla-maestra-y-cálculo-del-uqi)

In [36]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point

**CARGA DEL GEOJSON DE BARRIOS**

Es la capa base espacial del proyecto. 
Se usa como referencia para hacer joins espaciales.

Normalización de nombres para joins consistentes. 

In [37]:
gdf_barrios = gpd.read_file('barris.geojson')

gdf_barrios['nombre'] = gdf_barrios['nombre'].str.strip().str.lower()  #todos en minúscula

print(f"Barrios disponibles: {len(gdf_barrios)}")
print("Columnas:", gdf_barrios.columns.tolist())
display(gdf_barrios.head(3))

Barrios disponibles: 88
Columnas: ['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry']


,objectid,codbarrio,nombre,coddistbar,coddistrit,gis.gis.BARRIOS.area,geometry
0,63,3,la creu coberta,093,9,3.748875e+06,"POLYGON ((-0.38124 39.45463, -0.38416 39.45502..."
1,65,4,la fonteta s.lluis,104,10,2.388267e+06,"POLYGON ((-0.36941 39.44772, -0.36934 39.44759..."
2,601,5,rafalell-vistabella,175,17,NaN,"POLYGON ((-0.28767 39.55682, -0.2882 39.55711,..."


---
## 1. Calidad del aire

Dataset con mediciones horarias de 12 estaciones en Valencia (2016–2021).  
Construimos un `Air_Quality_Score` por estación agregando los cuatro contaminantes principales.

In [38]:
df_aire = pd.read_csv('calidad_aire.csv', sep=';')

print(f"Filas: {df_aire.shape[0]:,} | Columnas: {df_aire.shape[1]}")
print("\nColumnas disponibles:")
print(df_aire.columns.tolist())
print("\nPorcentaje de nulos por variable:")
display(df_aire.isnull().mean().mul(100).sort_values(ascending=False).round(1))

Filas: 449,026 | Columnas: 27

Columnas disponibles:
['Fecha', 'Día de la semana', 'Día del mes', 'Hora', 'Estación', 'PM1', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'O3', 'SO2', 'CO', 'Velocidad del viento', 'Dirección del viento', 'NH3', 'C7H8', 'C6H6', 'Ruido', 'C8H10', 'Temperatura', 'Humedad relativa', 'Presión', 'Radiación', 'Precipitación', 'Velocidad máxima del viento']

Porcentaje de nulos por variable:


C6H6                           89.7
C7H8                           89.6
C8H10                          89.5
NH3                            89.4
Ruido                          88.5
PM1                            83.3
CO                             66.4
PM2.5                          50.5
PM10                           50.5
SO2                            27.8
O3                             27.4
NO                             25.3
NOx                            25.3
NO2                            21.4
Humedad relativa                7.1
Temperatura                     3.7
Radiación                       3.6
Velocidad del viento            3.5
Velocidad máxima del viento     3.5
Precipitación                   3.2
Presión                         3.1
Dirección del viento            3.0
Estación                        0.0
Fecha                           0.0
Hora                            0.0
Día de la semana                0.0
Día del mes                     0.0
dtype: float64

**CONVERSIÓN DE FECHA Y EXTRACCIÓN DEL AÑO**

errors='coerce' convierte fechas mal formateadas a NaT en lugar de lanzar un error.

In [39]:
df_aire['Fecha'] = pd.to_datetime(df_aire['Fecha'], errors='coerce')
df_aire['Año'] = df_aire['Fecha'].dt.year

print("Registros por año:")
display(df_aire['Año'].value_counts().sort_index())

Registros por año:


Año
2016    61488
2017    70080
2018    63178
2019    70080
2020    87840
2021    96360
Name: count, dtype: int64

**ESTACIONES DISPONIBLES**

Identificamos qué estaciones existen antes de filtrar.


In [40]:
estaciones = df_aire['Estación'].unique()
print(f"Total de estaciones: {len(estaciones)}")
print(estaciones)

Total de estaciones: 12
<StringArray>
[            'Avda. Francia',              'Bulevard Sud',
              'Molí del Sol',               'Pista Silla',
               'Politécnico',           'Valencia Centro',
                   'Viveros',           'Puerto Valencia',
             'Nazaret Meteo',         'Consellería Meteo',
 'Puerto Moll Trans. Ponent',   'Puerto llit antic Túria']
Length: 12, dtype: str


**ELIMINACIÓN DE ESTACIONES METEOROLÓGICAS**

'Consellería Meteo' y 'Nazaret Meteo' solo registran variables climáticas (viento, temperatura…), no contaminantes.

Las descartamos para no introducir filas vacías en el score.

In [41]:
estaciones_descartar = ['Consellería Meteo', 'Nazaret Meteo']

df_aire = df_aire[~df_aire['Estación'].isin(estaciones_descartar)]

print(f"Estaciones restantes: {df_aire['Estación'].nunique()}")

Estaciones restantes: 10


**AGREGACIÓN: MEDIA HISTÓRICA POR ESTACIÓN**

No usamos datos horarios directamente; calculamos la media histórica (2016–2021) de cada contaminante por estación.

Esto captura el comportamiento estructural de cada zona.

Contaminantes seleccionados y su relevancia:
- PM2.5: partículas finas, muy dañinas para la salud
- PM10: partículas gruesas, impacto moderado
- NO2: tráfico rodado, marcador de zonas urbanas densas
- O3: ozono, formado por reacción fotoquímica

In [42]:
contaminantes = ['PM2.5', 'PM10', 'NO2', 'O3']

df_air_station = (
    df_aire
    .groupby('Estación')[contaminantes]
    .mean()
    .reset_index()
)

display(df_air_station)

,Estación,PM2.5,PM10,NO2,O3
0,Avda. Francia,9.931744,16.855239,23.556578,51.770598
1,Bulevard Sud,NaN,NaN,29.318896,50.490204
2,Molí del Sol,15.264481,17.113246,19.175958,50.512573
3,Pista Silla,10.880189,21.272138,31.159071,46.182419
4,Politécnico,10.960002,16.222207,16.803542,54.515141
5,Puerto Moll Trans. Ponent,10.357957,21.570932,26.698166,59.278504
6,Puerto Valencia,5.716549,8.993838,27.484174,44.908008
7,Puerto llit antic Túria,10.114264,19.582524,23.788653,49.015820
8,Valencia Centro,12.713658,20.461378,25.518310,NaN
9,Viveros,NaN,NaN,22.463383,53.512981


**IMPUTACIÓN DE VALORES NULOS**

Algunas estaciones no miden todos los contaminantes.

Imputamos con la media global para conservar la información espacial de todas las estaciones en el score final.

In [43]:
df_air_station[contaminantes] = df_air_station[contaminantes].fillna(
    df_air_station[contaminantes].mean()
)

print("Nulos restantes:", df_air_station[contaminantes].isnull().sum().sum())

Nulos restantes: 0


**NORMALIZACIÓN Y CÁLCULO DEL Air_Quality_Score**

Paso 1 – MinMax scaling: lleva cada contaminante al rango [0, 1]
   donde 0 = estación más contaminada, 1 = más limpia.

Paso 2 – Score ponderado (0–100):
   Invertimos la escala (1 - valor) para que puntuaciones altas
   representen MEJOR calidad del aire.

Pesos asignados según impacto en salud:
- PM2.5 → 0.35  (mayor riesgo cardiovascular/respiratorio)
- NO2   → 0.30  (indicador de densidad de tráfico)
- PM10  → 0.20  (impacto moderado)
- O3    → 0.15  (contaminante secundario, más variable)

In [44]:
scaler = MinMaxScaler()
df_air_norm = df_air_station.copy()
df_air_norm[contaminantes] = scaler.fit_transform(df_air_station[contaminantes])

df_air_norm['Air_Quality_Score'] = (
    0.35 * (1 - df_air_norm['PM2.5']) +
    0.30 * (1 - df_air_norm['NO2'])   +
    0.20 * (1 - df_air_norm['PM10'])  +
    0.15 * (1 - df_air_norm['O3'])
) * 100

df_air_final = (
    df_air_norm[['Estación', 'Air_Quality_Score']]
    .sort_values('Air_Quality_Score', ascending=False)
    .reset_index(drop=True)
)

print("Air_Quality_Score por estación:")
display(df_air_final)

Air_Quality_Score por estación:


,Estación,Air_Quality_Score
0,Puerto Valencia,77.679752
1,Politécnico,59.256524
2,Avda. Francia,50.771543
3,Puerto llit antic Túria,48.156029
4,Viveros,46.828851
5,Molí del Sol,41.280659
6,Bulevard Sud,35.657479
7,Valencia Centro,31.406552
8,Pista Silla,30.216470
9,Puerto Moll Trans. Ponent,27.308266


**ASIGNACIÓN DEL Air_Quality_Score A BARRIOS**
 
 La red de vigilancia de Valencia cuenta con 10 estaciones
 de medición, insuficientes para cubrir los 88 barrios de
 forma individual. Para asignar el score de cada estación
 al barrio correspondiente se utiliza un join espacial por
 distancia mínima (sjoin_nearest): cada barrio recibe el
 Air_Quality_Score de la estación más cercana a su centroide.

 Las coordenadas de las estaciones se obtienen del dataset
 "Estacions de mesurament automàtic de contaminació atmosfèrica"
 disponible en opendata.vlci.valencia.es, que incluye la
 ubicación georreferenciada de cada cabina de medición.

 Limitación metodológica: este enfoque asume que la calidad
 del aire de una estación es representativa de su entorno
 inmediato, lo cual es una simplificación. Barrios alejados
 de cualquier estación heredan el score de la más cercana
 aunque sus condiciones reales puedan diferir. Esta limitación
 es inherente a la densidad de la red de medición municipal
 y no puede resolverse sin datos adicionales.

In [45]:
gdf_estaciones = gpd.read_file('ubicacio_estacions.geojson')
print(f"Total estaciones: {len(gdf_estaciones)}")
print("\nColumnas:", gdf_estaciones.columns.tolist())
display(gdf_estaciones.head())

Total estaciones: 11

Columnas: ['objectid', 'nombre', 'direccion', 'tipozona', 'parametros', 'mediciones', 'so2', 'no2', 'o3', 'co', 'pm10', 'pm25', 'tipoemisio', 'fecha_carg', 'calidad_am', 'fiwareid', 'geometry']


,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,geometry
0,20,Cabanyal,CABANYAL,Urbana,NaN,NaN,NaN,32,NaN,NaN,26.0,8.0,Fondo,1781384400000,Razonablemente Buena,A09_CABANYAL_60m,POINT (-0.32853 39.47439)
1,21,Olivereta,OLIVERETA,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",NaN,NaN,39,NaN,NaN,21.0,8.0,Tráfico,1781384400000,Razonablemente Buena,A10_OLIVERETA_60m,POINT (-0.40592 39.46924)
2,14,Boulevar Sur,BULEVARD SUD,Urbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,1.0,35,82.0,NaN,NaN,NaN,Tráfico,1781384400000,Razonablemente Buena,A02_BULEVARDSUD_60m,POINT (-0.39634 39.4504)
3,22,Patraix,PATRAIX,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",NaN,NaN,51,NaN,NaN,22.0,8.0,Tráfico,1781384400000,Razonablemente Buena,A11_PATRAIX_60m,POINT (-0.40141 39.45919)
4,17,Universidad Politécnica,POLITÈCNIC,Suburbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,2.0,24,79.0,NaN,9.0,5.0,Fondo,1781384400000,Razonablemente Buena,A05_POLITECNIC_60m,POINT (-0.3374 39.47964)


In [46]:
# comprobar que las estaciones se llaman igual en el GeoJSON y en el df_air_final
print("Nombres en ubicacio_estacions:")
print(sorted(gdf_estaciones['nombre'].tolist()))

print("\nNombres en df_air_final:")
print(sorted(df_air_final['Estación'].tolist()))

Nombres en ubicacio_estacions:
['Boulevar Sur', 'Cabanyal', 'Centro', 'Dr. Lluch', 'Francia', 'Molí del Sol', 'Olivereta', 'Patraix', 'Pista de Silla', 'Universidad Politécnica', 'Viveros']

Nombres en df_air_final:
['Avda. Francia', 'Bulevard Sud', 'Molí del Sol', 'Pista Silla', 'Politécnico', 'Puerto Moll Trans. Ponent', 'Puerto Valencia', 'Puerto llit antic Túria', 'Valencia Centro', 'Viveros']


No coinciden casi ninguno. Vamos a hacer un mapeo manual. Además hay estaciones que están en uno pero no en el otro (Cabanyal, Dr. Lluch, Olivereta, Patraix son nuevas y no tienen histórico de aire; los tres del Puerto no tienen ubicación en el GeoJSON).

7 estaciones con match, 3 del Puerto sin ubicación. Esto es suficiente para hacer un buen join espacial. Los 3 del Puerto los descartamos del join ya que están fuera del área urbana de barrios de todas formas.

In [47]:
# Mapeo manual de nombres entre ambos datasets
nombre_map = {
    'Avda. Francia'           : 'Francia',
    'Bulevard Sud'            : 'Boulevar Sur',
    'Molí del Sol'            : 'Molí del Sol',
    'Pista Silla'             : 'Pista de Silla',
    'Politécnico'             : 'Universidad Politécnica',
    'Valencia Centro'         : 'Centro',
    'Viveros'                 : 'Viveros'
}

df_air_final['nombre_geo'] = df_air_final['Estación'].map(nombre_map)

# Unimos con las coordenadas del GeoJSON
gdf_aire = gdf_estaciones[['nombre', 'geometry']].merge(
    df_air_final.dropna(subset=['nombre_geo']),
    left_on='nombre',
    right_on='nombre_geo',
    how='inner'
)

gdf_aire = gpd.GeoDataFrame(gdf_aire, geometry='geometry', crs=gdf_barrios.crs)

print(f"Estaciones con score y coordenadas: {len(gdf_aire)}")
display(gdf_aire[['nombre', 'Air_Quality_Score', 'geometry']])

Estaciones con score y coordenadas: 7


,nombre,Air_Quality_Score,geometry
0,Boulevar Sur,35.657479,POINT (-0.39634 39.4504)
1,Universidad Politécnica,59.256524,POINT (-0.3374 39.47964)
2,Viveros,46.828851,POINT (-0.36965 39.47964)
3,Molí del Sol,41.280659,POINT (-0.40881 39.48111)
4,Pista de Silla,30.216470,POINT (-0.37664 39.45806)
5,Centro,31.406552,POINT (-0.3764 39.47055)
6,Francia,50.771543,POINT (-0.34299 39.45783)


**JOIN ESPACIAL POR DISTANCIA: AIRE → BARRIOS**

Cada barrio recibe el Air_Quality_Score de la estación más cercana a su centroide mediante sjoin_nearest.

Usamos los centroides de los barrios como punto de referencia para calcular la distancia a cada estación.

In [48]:
gdf_barrios_centroid = gdf_barrios.copy()
gdf_barrios_centroid['geometry'] = gdf_barrios.geometry.centroid

gdf_aire_barrios = gpd.sjoin_nearest(
    gdf_barrios_centroid[['geometry', 'nombre']],
    gdf_aire[['geometry', 'Air_Quality_Score']],
    how='left'
)

df_aire_final = (
    gdf_aire_barrios[['nombre', 'Air_Quality_Score']]
    .rename(columns={'nombre': 'barrio'})
    .reset_index(drop=True)
)

print(f"Barrios con Air_Quality_Score: {len(df_aire_final)}")
print("\nDistribución del score:")
display(df_aire_final.sort_values('Air_Quality_Score', ascending=False).head(10))

Barrios con Air_Quality_Score: 88

Distribución del score:


C:\Users\Yaiza\AppData\Local\Temp\ipykernel_11992\752585675.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_barrios_centroid['geometry'] = gdf_barrios.geometry.centroid
c:\Users\Yaiza\OneDrive - UPV\3_TERCERO\CUATRI_B\EDM\PROYECTO\proyEDM\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


,barrio,Air_Quality_Score
3,cami de vera,59.256524
2,rafalell-vistabella,59.256524
43,cabanyal-canyamelar,59.256524
44,la carrasca,59.256524
46,la vega baixa,59.256524
45,la malva-rosa,59.256524
48,ciutat jardi,59.256524
26,l'illa perduda,59.256524
41,l'amistat,59.256524
42,betero,59.256524


In [49]:
print(f"Barrios con Air_Quality_Score: {df_aire_final['Air_Quality_Score'].notna().sum()}")

# Cuántos barrios fueron asignados a cada estación
print("\nBarrios asignados por estación:")
display(
    gdf_aire_barrios.groupby('Air_Quality_Score')
    .size()
    .reset_index(name='num_barrios')
    .merge(gdf_aire[['Air_Quality_Score', 'nombre']], on='Air_Quality_Score')
    .sort_values('num_barrios', ascending=False)
)

Barrios con Air_Quality_Score: 88

Barrios asignados por estación:


,Air_Quality_Score,num_barrios,nombre
4,46.828851,18,Viveros
5,50.771543,14,Francia
2,35.657479,13,Boulevar Sur
3,41.280659,13,Molí del Sol
1,31.406552,11,Centro
6,59.256524,11,Universidad Politécnica
0,30.216470,8,Pista de Silla


---
## 2. Quejas ciudadanas

Dataset de incidencias y quejas registradas en el Ayuntamiento (~90.000 registros).  
Variable objetivo: `num_quejas` por barrio, proxy de problemas urbanos percibidos.

In [50]:
df_quejas = pd.read_csv('quejas.csv', sep=';')

print(f"Filas: {df_quejas.shape[0]:,} | Columnas: {df_quejas.shape[1]}")
print("\nColumnas:")
print(df_quejas.columns.tolist())
print("\nNulos por columna:")
display(df_quejas.isnull().sum().sort_values(ascending=False))

Filas: 90,005 | Columnas: 13

Columnas:
['tipo_solicitud', 'canal_entrada', 'fecha_entrada_ayuntamiento', 'tema', 'subtema', 'distrito_solicitante', 'barrio_solicitante', 'distrito_localización', 'barrio_localización', 'distrito_solicitante_código', 'barrio_solicitante_código', 'distrito_localización_código', 'barrio_localización_código']

Nulos por columna:


barrio_localización             2
barrio_localización_código      2
fecha_entrada_ayuntamiento      0
canal_entrada                   0
tipo_solicitud                  0
subtema                         0
tema                            0
barrio_solicitante              0
distrito_solicitante            0
distrito_localización           0
distrito_solicitante_código     0
barrio_solicitante_código       0
distrito_localización_código    0
dtype: int64

In [51]:
# conversión de fechas

df_quejas['fecha_entrada_ayuntamiento'] = pd.to_datetime(
    df_quejas['fecha_entrada_ayuntamiento'],
    errors='coerce'
)

print(f"Fechas no convertibles (NaT): {df_quejas['fecha_entrada_ayuntamiento'].isna().sum():,}")

Fechas no convertibles (NaT): 54,349


**ELIMINACIÓN DE FILAS SIN FECHA NI BARRIO**

Sin fecha no podemos hacer análisis temporal.

Sin barrio no podemos asignar la queja a ningún polígono.

Ambos campos son imprescindibles; descartamos las filas afectadas.

In [52]:
df_quejas = df_quejas.dropna(subset=['fecha_entrada_ayuntamiento', 'barrio_localización'])

print(f"Registros tras limpieza: {len(df_quejas):,}")

Registros tras limpieza: 35,655


**ELIMINACIÓN DE BARRIOS NO GEORREFERENCIABLES**

Valores que no se pueden asociar a ningún polígono del GeoJSON:
 'desconocido', 'No hi consta', 'no consta',
 'Fora de València', 'Fuera de València'


In [53]:
valores_invalidos = [
    'desconocido',
    'No hi consta',
    'no consta',
    'Fora de València',
    'Fuera de València',
    'No consta'
]

df_quejas = df_quejas[
    ~df_quejas['barrio_localización'].isin(valores_invalidos)
]

# Normalizamos para que coincida con gdf_barrios['barrio']
df_quejas['barrio_localización'] = df_quejas['barrio_localización'].str.strip().str.lower()

print(f"Registros con barrio válido: {len(df_quejas):,}")

Registros con barrio válido: 29,202


**COMPROBACIÓN SISTEMÁTICA DE BARRIOS SIN MATCH**

En lugar de buscar caso por caso, comparamos directamente
 los nombres de barrio en quejas vs en el GeoJSON de barrios
 para detectar TODAS las discrepancias restantes de una vez.


In [54]:
barrios_quejas  = set(df_quejas['barrio_localización'].unique())
barrios_geojson = set(gdf_barrios['nombre'].unique())

sin_match = barrios_quejas - barrios_geojson

print(f"Nombres en quejas sin match en el GeoJSON: {len(sin_match)}")
print(sorted(sin_match))

Nombres en quejas sin match en el GeoJSON: 23
['benimàmet', 'beteró', 'borbotó', 'camí de vera', 'camí fondo', 'camí real', 'cases de bàrcena', 'ciutat de les arts i de les ciències', 'ciutat jardí', 'ciutat universitària', 'el botànic', 'el cabanyal-el canyamelar', "el castellar-l'oliverar", 'en dependencias municipales', 'exposició', 'fonteta de sant lluís', 'gran via', 'la bega baixa', 'mauella', 'mont-olivet', 'orriols', 'sant llorenç', 'sant marcel·lí']


**CORRECCIÓN DE NOMBRES PARA JOIN CON GEOJSON**

El dataset de quejas y el GeoJSON de barrios no comparten
 exactamente los mismos nombres: acentos valencianos, artículos,
 puntos medios y caracteres especiales generan discrepancias
 que rompen el join. Se corrigen TODOS los casos detectados
 mediante comparación sistemática de conjuntos (sets), no solo
los encontrados manualmente.


In [ ]:
nombre_fix = {
    'el cabanyal-el canyamelar'           : 'cabanyal-canyamelar',
    'el botànic'                          : 'el botanic',
    'sant llorenç'                        : 'sant llorens',
    'ciutat universitària'                : 'ciutat universitaria',
    'ciutat jardí'                        : 'ciutat jardi',
    'ciutat de les arts i de les ciències': 'ciutat de les arts i de les ciencies',
    'sant marcel·lí'                      : 'sant marcel.li',
    'orriols'                             : 'els orriols',
    'benimàmet'                           : 'benimamet',
    'beteró'                              : 'betero',
    'borbotó'                             : 'borboto',
    'camí de vera'                        : 'cami de vera',
    'camí fondo'                          : 'cami fondo',
    'camí real'                           : 'cami real',
    "cases de bàrcena"                    : 'les cases de barcena',
    "el castellar-l'oliverar"             : "castellar-l'oliveral",
    'exposició'                           : 'exposicio',
    'fonteta de sant lluís'               : 'la fonteta s.lluis',
    'gran via'                            : 'la gran via',
    'la bega baixa'                       : 'la vega baixa',
    'mauella'                             : 'mahuella-tauladella',
    'mont-olivet'                         : 'montolivet',
}

df_quejas['barrio_localización'] = df_quejas['barrio_localización'].replace(nombre_fix)

# 'en dependencias municipales' no es un barrio real, es un valor administrativo, lo eliminamos 
df_quejas = df_quejas[df_quejas['barrio_localización'] != 'en dependencias municipales']

In [56]:
barrios_quejas  = set(df_quejas['barrio_localización'].unique())
barrios_geojson = set(gdf_barrios['nombre'].unique())
sin_match = barrios_quejas - barrios_geojson
print(f"Sin match restantes: {len(sin_match)}")

Sin match restantes: 0


In [57]:
# extracción del año

df_quejas['año'] = df_quejas['fecha_entrada_ayuntamiento'].dt.year

print("Distribución de quejas por año:")
display(df_quejas['año'].value_counts().sort_index())

Distribución de quejas por año:


año
2020    4393
2021    4835
2022    5100
2023    4502
2024    4383
2025    4584
2026    1034
Name: count, dtype: int64

**AGREGACIONES PRINCIPALES**

Quejas totales por barrio (input principal del UQI)

In [58]:
df_quejas_barrio = (
    df_quejas
    .groupby('barrio_localización')
    .size()
    .reset_index(name='num_quejas')
    .sort_values('num_quejas', ascending=False)
)

print("Top 10 barrios con más quejas:")
display(df_quejas_barrio.head(10))

Top 10 barrios con más quejas:


,barrio_localización,num_quejas
3,benicalap,1192
73,russafa,1046
60,malilla,996
2,arrancapins,977
43,la creu del grau,929
69,patraix,899
48,la malva-rosa,812
49,la petxina,741
6,benimaclet,738
68,nou moles,715


In [59]:
# Normalizamos para que coincida con gdf_barrios['barrio']
df_quejas['distrito_localización'] = df_quejas['distrito_localización'].str.strip().str.lower()

# Quejas por distrito (visión agregada)
df_quejas_distrito = (
    df_quejas
    .groupby('distrito_localización')
    .size()
    .reset_index(name='num_quejas')
    .sort_values('num_quejas', ascending=False)
)

print("Quejas por distrito:")
display(df_quejas_distrito.head(10))

# Matriz barrio × tema (útil para clustering)
df_barrio_tema = (
    df_quejas
    .groupby(['barrio_localización', 'tema'])
    .size()
    .reset_index(name='conteo')
)

print("\nEjemplo matriz barrio-tema:")
display(df_barrio_tema.head())

# Evolución temporal
df_quejas_temporal = (
    df_quejas
    .groupby('año')
    .size()
    .reset_index(name='num_quejas')
)

print("\nEvolución anual de quejas:")
display(df_quejas_temporal)

Quejas por distrito:


,distrito_localización,num_quejas
17,quatre carreres,2740
7,extramurs,2385
3,camins al grau,2366
13,poblats marítims,2257
5,ciutat vella,2138
12,patraix,1843
9,l'eixample,1654
10,l'olivereta,1507
0,algirós,1446
8,jesús,1441



Ejemplo matriz barrio-tema:


,barrio_localización,tema,conteo
0,aiora,Agradecimientos,6
1,aiora,Atención Personal Municipal,9
2,aiora,COVID-19,5
3,aiora,Contaminación acústica,21
4,aiora,Discrepancias con actuaciones municipales,39



Evolución anual de quejas:


,año,num_quejas
0,2020,4393
1,2021,4835
2,2022,5100
3,2023,4502
4,2024,4383
5,2025,4584
6,2026,1034


**NORMALIZACIÓN: Complaints_Score**

Invertimos la escala: más quejas → peor puntuación.

Score 100 = barrio con menos quejas.

In [88]:
scaler_q = MinMaxScaler()
df_quejas_barrio['Complaints_Score'] = (
    1 - scaler_q.fit_transform(df_quejas_barrio[['num_quejas']])
) * 100

df_quejas_final = df_quejas_barrio[['barrio_localización', 'num_quejas', 'Complaints_Score']].copy()

print("Dataset final de quejas:")
display(df_quejas_final.head(10))

Dataset final de quejas:


,barrio_localización,num_quejas,Complaints_Score
3,benicalap,1192,1.110223e-14
73,russafa,1046,1.231029e+01
60,malilla,996,1.652614e+01
2,arrancapins,977,1.812816e+01
43,la creu del grau,929,2.217538e+01
69,patraix,899,2.470489e+01
48,la malva-rosa,812,3.204047e+01
49,la petxina,741,3.802698e+01
6,benimaclet,738,3.827993e+01
68,nou moles,715,4.021922e+01


**TRATAMIENTO DE BARRIOS SIN QUEJAS REGISTRADAS**

A diferencia de otros casos, aquí la ausencia de quejas es un dato real
 (positivo, no limitación de cobertura), por lo que asignamos
 el score máximo de 0 quejas: Complaints_Score = 100.

In [ ]:
todos_los_barrios = pd.DataFrame({'barrio_localización': gdf_barrios['nombre']})
df_quejas_final = todos_los_barrios.merge(df_quejas_final, on='barrio_localización', how='left')

df_quejas_final['num_quejas']       = df_quejas_final['num_quejas'].fillna(0)
df_quejas_final['Complaints_Score'] = df_quejas_final['Complaints_Score'].fillna(100)

print(f"Total barrios: {len(df_quejas_final)}")
print(f"Barrios sin queja alguna: {df_quejas_final['num_quejas'].eq(0).sum()}")

Total barrios: 88
Barrios sin queja alguna: 1


---
## 3. Ruido urbano

Tres capas GeoJSON con niveles de ruido en dB(A): diurno, vespertino y nocturno.  
Construimos un `Noise_Score` por barrio ponderando las tres franjas.

**CARGA DE LAS TRES CAPAS DE RUIDO**

Cada archivo representa una franja horaria del mapa de ruido
estratégico de Valencia (Directiva Europea 2002/49/CE).

In [61]:
gdf_ruido_dia  = gpd.read_file('mapa_soroll_dia.geojson')
gdf_ruido_vesp = gpd.read_file('mapa_soroll_vesprada.geojson')
gdf_ruido_nit  = gpd.read_file('mapa_soroll_nit.geojson')

print("Columnas disponibles (día):")
print(gdf_ruido_dia.columns.tolist())
print("\nMuestra:")
display(gdf_ruido_dia.head(3))

Columnas disponibles (día):
['objectid', 'gridcode', 'geometry']

Muestra:


,objectid,gridcode,geometry
0,1,1,"MULTIPOLYGON (((-0.31916 39.31273, -0.31913 39..."
1,2,1,"MULTIPOLYGON (((-0.28622 39.29487, -0.28624 39..."
2,3,1,"MULTIPOLYGON (((-0.41335 39.44415, -0.41344 39..."


In [62]:
# Verificamos que todas las capas comparten el mismo CRS
assert gdf_barrios.crs == gdf_ruido_dia.crs, "CRS no coinciden"
print(f"CRS confirmado: {gdf_barrios.crs}")

CRS confirmado: EPSG:4326


**IDENTIFICAR Y CONVERTIR COLUMNA DE NIVEL DE RUIDO**

Los archivos de ruido contienen 'gridcode', un código entero
 que representa rangos de dB(A) según la Directiva Europea
 de ruido ambiental 2002/49/CE (intervalos de 5 dB).

Valores presentes en Valencia (1–6):
- 1 → 35–40 dB  (silencioso)
- 2 → 40–45 dB  (tranquilo)
- 3 → 45–50 dB  (moderado)
- 4 → 50–55 dB  (moderado-alto)
- 5 → 55–60 dB  (alto)
- 6 → 60–65 dB  (muy alto)

 Usamos el punto medio de cada intervalo como valor numérico
 representativo para poder operar con él (media, normalización…).

In [63]:
db_col = 'gridcode'

gridcode_to_db = {1: 37.5, 2: 42.5, 3: 47.5, 4: 52.5, 5: 57.5, 6: 62.5}

gdf_ruido_dia['db_valor']  = gdf_ruido_dia['gridcode'].map(gridcode_to_db)
gdf_ruido_vesp['db_valor'] = gdf_ruido_vesp['gridcode'].map(gridcode_to_db)
gdf_ruido_nit['db_valor']  = gdf_ruido_nit['gridcode'].map(gridcode_to_db)

print(f"Valores nulos tras mapeo: {gdf_ruido_dia['db_valor'].isna().sum()}")
display(gdf_ruido_dia[['objectid', 'gridcode', 'db_valor']].head())

Valores nulos tras mapeo: 0


,objectid,gridcode,db_valor
0,1,1,37.5
1,2,1,37.5
2,3,1,37.5
3,4,1,37.5
4,5,1,37.5


**JOIN ESPACIAL: RUIDO → BARRIOS**

Asignamos cada polígono de ruido al barrio que lo contiene
 usando un join espacial 'within' (el centroide del polígono
 de ruido cae dentro del polígono del barrio).


In [64]:
def ruido_por_barrio(gdf_ruido, gdf_barrios, col_nivel, nombre_col_out):
    """Calcula el nivel medio de ruido por barrio vía join espacial."""
    # Extraemos el punto central de cada zona de ruido
    gdf_ruido = gdf_ruido.copy()
    gdf_ruido['geometry'] = gdf_ruido.geometry.centroid

    joined = gpd.sjoin(
        gdf_ruido[['geometry', col_nivel]],
        gdf_barrios[['geometry', 'nombre']],  # <- ajustar 'nombre' si difiere
        how='left',
        predicate='within'
    )

    result = (
        joined
        .groupby('nombre')[col_nivel]
        .mean()
        .reset_index()
        .rename(columns={col_nivel: nombre_col_out})
    )
    return result

df_ruido_dia  = ruido_por_barrio(gdf_ruido_dia,  gdf_barrios, 'db_valor', 'db_dia')
df_ruido_vesp = ruido_por_barrio(gdf_ruido_vesp, gdf_barrios, 'db_valor', 'db_vesprada')
df_ruido_nit  = ruido_por_barrio(gdf_ruido_nit,  gdf_barrios, 'db_valor', 'db_nit')

df_ruido = (
    df_ruido_dia
    .merge(df_ruido_vesp, on='nombre', how='outer')
    .merge(df_ruido_nit,  on='nombre', how='outer')
)

display(df_ruido.head())

C:\Users\Yaiza\AppData\Local\Temp\ipykernel_11992\3240910460.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_ruido['geometry'] = gdf_ruido.geometry.centroid
C:\Users\Yaiza\AppData\Local\Temp\ipykernel_11992\3240910460.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_ruido['geometry'] = gdf_ruido.geometry.centroid
C:\Users\Yaiza\AppData\Local\Temp\ipykernel_11992\3240910460.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_ruido['geometry'] = gdf_ruido.geometry.centroid


,nombre,db_dia,db_vesprada,db_nit
0,arrancapins,NaN,47.5,NaN
1,benicalap,57.5,62.5,50.833333
2,benimaclet,NaN,42.5,NaN
3,campanar,57.5,57.5,57.500000
4,carpesa,47.5,42.5,37.500000


El resultado muestra el nivel medio de ruido en dB(A) por barrio.
Los decimales largos (p.ej. 50.833…) son normales: indican que
dentro del barrio coexisten polígonos con distintos niveles de ruido,
 y el valor es su media. Los NaN corresponden a barrios sin polígonos
 mapeados en esa franja horaria (sin datos en la fuente original).

**CÁLCULO DEL Noise_Score**

Combinamos las tres franjas con pesos:
- Día       → 0.40 (mayor exposición acumulada)
- Vespertino → 0.35
- Noche     → 0.25 (más sensible para la salud, pero menor duración)

Invertimos la escala para que Score 100 = barrio más silencioso.


In [65]:
df_ruido['db_ponderado'] = (
    0.40 * df_ruido['db_dia'] +
    0.35 * df_ruido['db_vesprada'] +
    0.25 * df_ruido['db_nit']
)

scaler_r = MinMaxScaler()
df_ruido['Noise_Score'] = (
    1 - scaler_r.fit_transform(df_ruido[['db_ponderado']])
) * 100

df_ruido_final = df_ruido[['nombre', 'Noise_Score']].rename(columns={'nombre': 'barrio'})

print("Noise_Score por barrio (muestra):")
display(df_ruido_final.sort_values('Noise_Score', ascending=False).head(10))

Noise_Score por barrio (muestra):


,barrio,Noise_Score
16,la fonteta s.lluis,100.000000
26,mestalla,97.058824
12,el saler,94.823529
4,carpesa,90.588235
7,ciutat fallera,82.941176
21,la xerea,82.352941
35,russafa,77.843137
41,trinitat,74.117647
11,el pla del remei,73.529412
28,morvedre,70.588235


Los resultados tienen un problema claro: La Fonteta S.Lluis, El Saler y Carpesa con scores altísimos de silencio no cuadran con la realidad urbana de Valencia, y hay barrios céntricos muy ruidosos como Russafa o Mestalla que deberían estar más abajo.
El problema probable es el de los NaN que se imputaron con la mediana antes de normalizar. Los barrios sin polígonos mapeados (como veíamos antes) recibieron un valor medio artificial que los disparó hacia arriba en el ranking.

In [66]:
print(df_ruido_final.shape)
print(df_ruido_final['Noise_Score'].isna().sum())
display(df_ruido_final.head())

(43, 2)
19


,barrio,Noise_Score
0,arrancapins,NaN
1,benicalap,23.137255
2,benimaclet,NaN
3,campanar,23.529412
4,carpesa,90.588235


In [67]:
# Cuántos barrios tienen NaN en cada franja antes de imputar
print(df_ruido['db_dia'].isna().sum())
print(df_ruido['db_vesprada'].isna().sum())
print(df_ruido['db_nit'].isna().sum())
print(f"\nTotal barrios: {len(df_ruido)}")

11
9
12

Total barrios: 43


Aqui tenemos un problema, dado que solo tenemos 43 barrios de los 88 que tiene Valencia, y además con bastantes NaN. El mapa de ruido no tiene cobertura completa de la ciudad

In [68]:
# Los 45 barrios que faltan

barrios_con_ruido = set(df_ruido['nombre'])
barrios_totales = set(gdf_barrios['nombre'])
print("Barrios sin ningún dato de ruido:")
print(barrios_totales - barrios_con_ruido)

Barrios sin ningún dato de ruido:
{'el mercat', 'betero', 'el pilar', 'sant antoni', 'beniferri', 'benimamet', 'la llum', 'cami fondo', 'la malva-rosa', 'tres forques', "l'hort de senabre", 'la roqueta', 'albors', 'safranar', 'sant isidre', 'el palmar', 'la creu del grau', 'patraix', 'borboto', "l'amistat", 'el botanic', 'nou moles', "l'illa perduda", 'faitanar', 'la fontsanta', 'els orriols', 'la creu coberta', 'la seu', 'ciutat jardi', 'el perellonet', 'mahuella-tauladella', 'cami de vera', 'cabanyal-canyamelar', 'soternes', 'el calvari', 'la vega baixa', 'el carme', 'sant llorens', 'massarrojos', 'benifaraig', 'la petxina', 'en corts', 'cami real', 'aiora', 'favara'}


**TRATAMIENTO DE BARRIOS SIN COBERTURA DE RUIDO**

El mapa de ruido municipal presenta dos tipos de ausencia:
  1. Barrios con fila pero sin dato en alguna franja (19 casos):
      su db_ponderado salió NaN porque faltaban franjas horarias.
  2. Barrios sin ningún registro en ninguna franja (45 casos):
      no tienen fila en df_ruido_final.

En ambos casos aplicamos la misma decisión metodológica:
score neutro de 50 (ni premiados ni penalizados).

Se documenta con un flag 'ruido_con_dato' para transparencia.

In [69]:
todos_los_barrios = pd.DataFrame({'barrio': gdf_barrios['nombre']})
df_ruido_final = todos_los_barrios.merge(df_ruido_final, on='barrio', how='left')

# Flag ANTES del fillna para distinguir dato real de imputado
df_ruido_final['ruido_con_dato'] = df_ruido_final['Noise_Score'].notna()

df_ruido_final['Noise_Score'] = df_ruido_final['Noise_Score'].fillna(50)

print(f"Total barrios             : {len(df_ruido_final)}")
print(f"Barrios con dato real     : {df_ruido_final['ruido_con_dato'].sum()}")
print(f"Barrios con score neutro  : {(~df_ruido_final['ruido_con_dato']).sum()}")

Total barrios             : 88
Barrios con dato real     : 24
Barrios con score neutro  : 64


El mapa de ruido municipal cubre únicamente 24 de los 88 barrios
de Valencia (27% del total). Los 64 restantes no tienen polígonos mapeados en ninguna de las tres franjas horarias, lo que refleja
 una limitación de la fuente original y no una ausencia de ruido.
 Estos barrios reciben un Noise_Score neutro de 50, documentado
mediante el flag 'ruido_con_dato' para que sea identificable
en análisis posteriores y en la app de Streamlit.

---
## 4. Equipamientos municipales

Puntos de servicios públicos: bibliotecas, centros cívicos, colegios, centros de salud, etc.  
Variable objetivo: densidad de equipamientos por barrio (`Services_Score`).

In [70]:
gdf_equip = gpd.read_file('equipaments_municipals.geojson')
gdf_equip = gdf_equip.to_crs(gdf_barrios.crs)

print(f"Total equipamientos: {len(gdf_equip):,}")
print("\nColumnas:")
print(gdf_equip.columns.tolist())
print("\nTipos de equipamiento disponibles:")
# Ajustar el nombre de columna si difiere
if 'tipo' in gdf_equip.columns:
    display(gdf_equip['tipo'].value_counts().head(15))
display(gdf_equip.head(3))

Total equipamientos: 2,000

Columnas:
['objectid', 'equipamien', 'x', 'y', 'identifica', 'shape', 'idclase', 'codvia', 'numportal', 'idsubclase', 'telefono', 'clase', 'geometry']

Tipos de equipamiento disponibles:


c:\Users\Yaiza\OneDrive - UPV\3_TERCERO\CUATRI_B\EDM\PROYECTO\proyEDM\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: OGRGeoJSONReadRawPoint(): Invalid coord dimension for '[ ]'. At least 2 dimensions must be present.
  return ogr_read(


,objectid,equipamien,x,y,identifica,shape,idclase,codvia,numportal,idsubclase,telefono,clase,geometry
0,20118,UNIVERSIDAD POPULAR - LA TORRE,724348.283580,4.368230e+06,006397,None,7,10290.0,S/N,No tiene,963361840.0,Instalaciones educativas,POINT (-0.39329 39.43442)
1,20157,ASOCIACIÓN RETO A LA ESPERANZA,725551.744379,4.373922e+06,001967,None,22,65850.0,31,No tiene,962112717.0,invisible,POINT (-0.37739 39.48534)
2,19987,UNIVERSIDAD POPULAR - EL PALMAR,731270.292194,4.354955e+06,006396,None,7,76260.0,1,No tiene,961620116.0,Instalaciones educativas,POINT (-0.31752 39.31311)


**LIMPIEZA DEL DATASET DE EQUIPAMIENTOS**

liminamos tres categorías que no aportan al análisis
de calidad urbana: 'invisible' y 'Sin clase Mapa' por
 falta de clasificación real, y 'Puntos de venta EMT'
 por ser infraestructura de venta de tickets, no servicios
 de proximidad para el ciudadano.

También eliminamos los 13 registros con geometría nula
 (coordenadas vacías en la fuente original).

In [71]:
clases_descartar = ['invisible', 'Sin clase Mapa', 'Puntos de venta EMT']

gdf_equip = gdf_equip[~gdf_equip['clase'].isin(clases_descartar)]
gdf_equip = gdf_equip[gdf_equip.geometry.notna()]

print(f"Equipamientos tras limpieza : {len(gdf_equip):,}")
print(f"\nClases restantes:")
display(gdf_equip['clase'].value_counts())

Equipamientos tras limpieza : 1,497

Clases restantes:


clase
Bienestar Social            703
Instalaciones educativas    481
Instalaciones deportivas     79
Instalaciones sanitarias     73
Oficinas municipales         52
Centros juveniles            26
Policía                      23
Museos                       21
Bibliotecas                  17
Teatros                      11
Mercados                      7
Alojamiento                   2
Correos                       1
Archivos                      1
Name: count, dtype: int64

**JOIN ESPACIAL: EQUIPAMIENTOS → BARRIOS**

Contamos cuántos equipamientos caen dentro de cada barrio.

In [72]:
gdf_equip_join = gpd.sjoin(
    gdf_equip[['geometry']],
    gdf_barrios[['geometry', 'nombre']],
    how='left',
    predicate='within'
)

df_equip_barrio = (
    gdf_equip_join
    .groupby('nombre')
    .size()
    .reset_index(name='num_equipamientos')
)

print("Top 10 barrios por número de equipamientos:")
display(df_equip_barrio.sort_values('num_equipamientos', ascending=False).head(10))

Top 10 barrios por número de equipamientos:


,nombre,num_equipamientos
74,sant francesc,55
10,cabanyal-canyamelar,49
23,el carme,46
6,benimaclet,44
55,la xerea,41
40,la carrasca,39
3,benicalap,38
29,el pilar,37
66,nou moles,36
34,exposicio,33


**CÁLCULO DEL Services_Score**

Más equipamientos → mejor puntuación.

Score 100 = barrio mejor dotado de servicios.

In [73]:
scaler_e = MinMaxScaler()
df_equip_barrio['Services_Score'] = (
    scaler_e.fit_transform(df_equip_barrio[['num_equipamientos']])
) * 100

df_equip_final = df_equip_barrio[['nombre', 'Services_Score']].rename(columns={'nombre': 'barrio'})

print("Services_Score por barrio (muestra):")
display(df_equip_final.sort_values('Services_Score', ascending=False).head(10))

Services_Score por barrio (muestra):


,barrio,Services_Score
74,sant francesc,100.000000
10,cabanyal-canyamelar,88.888889
23,el carme,83.333333
6,benimaclet,79.629630
55,la xerea,74.074074
40,la carrasca,70.370370
3,benicalap,68.518519
29,el pilar,66.666667
66,nou moles,64.814815
34,exposicio,59.259259


In [74]:
# revisamos que todos lo barrios tengan dato
print(f"Barrios con dato: {len(df_equip_final)}")

Barrios con dato: 85


**TRATAMIENTO DE BARRIOS SIN EQUIPAMIENTOS REGISTRADOS**

3 barrios no aparecen en el join espacial, probablemente
 barrios periféricos o rurales sin equipamientos municipales
 registrados en la fuente. Les asignamos score 0 ya que
la ausencia de equipamientos es un dato real, no una
 limitación de cobertura como ocurría con el ruido.

In [75]:
todos_los_barrios = pd.DataFrame({'barrio': gdf_barrios['nombre']})
df_equip_final = todos_los_barrios.merge(df_equip_final, on='barrio', how='left')

print("Barrios sin equipamientos registrados:")
print(df_equip_final[df_equip_final['Services_Score'].isna()]['barrio'].tolist())

df_equip_final['Services_Score'] = df_equip_final['Services_Score'].fillna(0)

print(f"\nTotal barrios    : {len(df_equip_final)}")
print(f"Con equipamientos: {df_equip_final['Services_Score'].gt(0).sum()}")
print(f"Sin equipamientos: {df_equip_final['Services_Score'].eq(0).sum()}")

Barrios sin equipamientos registrados:
['rafalell-vistabella', 'mahuella-tauladella', 'faitanar']

Total barrios    : 88
Con equipamientos: 83
Sin equipamientos: 5


In [76]:
print(df_equip_final[df_equip_final['Services_Score'].eq(0)]['barrio'].tolist())

['rafalell-vistabella', 'el perellonet', 'les cases de barcena', 'mahuella-tauladella', 'faitanar']


Tras el join espacial, 88 barrios quedan cubiertos pero 5
presentan Services_Score = 0. Se distinguen dos casos:

   - Sin registro en el join (3 barrios): Rafalell-Vistabella,
     Mahuella-Tauladella y Faitanar. Son pedanías rurales cuyos
     límites no intersectan con ningún punto del dataset.

   - Con join pero 0 equipamientos (2 barrios): El Perellonet y
     Les Cases de Barcena. Aparecen en el join espacial pero
     el ayuntamiento no tiene ningún equipamiento registrado
     dentro de sus límites.

 A diferencia del ruido, aquí NO asignamos score neutro (50)
 sino score 0, porque la ausencia de equipamientos es un dato
 real y verificable, no una limitación de cobertura del mapa.
 Penalizar estos barrios en el UQI es metodológicamente correcto.

---
## 5. Valenbisi (movilidad)

Estaciones del servicio de bicicleta pública de Valencia con ubicación y capacidad.  
Variable objetivo: accesibilidad al transporte sostenible por barrio (`Mobility_Score`).

In [77]:
gdf_valen = gpd.read_file('disponibilitat_valenbisi.geojson')
gdf_valen = gdf_valen.to_crs(gdf_barrios.crs)

print(f"Total estaciones Valenbisi: {len(gdf_valen)}")
print("\nColumnas:")
print(gdf_valen.columns.tolist())
display(gdf_valen.head(3))

Total estaciones Valenbisi: 273

Columnas:
['gid', 'name', 'number', 'address', 'open', 'available', 'free', 'total', 'ticket', 'updated_at', 'update_jcd', 'geometry']


,gid,name,number,address,open,available,free,total,ticket,updated_at,update_jcd,geometry
0,902338,209_AVDA_GASPAR_AGUILAR_ESQ_CALLE_MUSICO_PENELLA,209,Gaspar Aguilar - Músico Penella,T,0,0,18,F,17/01/2026 00:06:23,1768545672000,POINT (-0.39302 39.45208)
1,902229,097_AVDA. BLASCO IBAÑEZ 7,97,Blasco Ibañez 121,T,0,23,23,F,11/06/2026 10:50:36,1781167358000,POINT (-0.34313 39.47307)
2,902373,244_AVDA_PIO_BAROJA_VALLE_DE_LA_BALLESTERA,244,Valle de la Ballestera - Pio Baroja,T,12,8,20,F,11/06/2026 10:50:36,1781167768000,POINT (-0.40614 39.47851)


`available` y `free` son datos en tiempo real capturados en un momento concreto, no son representativos. Usaremos `total` como medida de capacidad estructural de cada estación.

**JOIN ESPACIAL: VALENBISI → BARRIOS**

Calculamos número de estaciones y capacidad total por barrio.

Usamos 'total' (anclajes por estación) como medida de capacidad
 estructural, ya que 'available' y 'free' son datos en tiempo
real capturados en un momento puntual y no son representativos.

In [78]:
gdf_valen = gdf_valen.to_crs(gdf_barrios.crs)

gdf_valen_join = gpd.sjoin(
    gdf_valen[['geometry', 'total']],
    gdf_barrios[['geometry', 'nombre']],
    how='left',
    predicate='within'
)

df_valen_barrio = (
    gdf_valen_join
    .groupby('nombre')
    .agg(
        num_estaciones=('total', 'count'),
        capacidad_total=('total', 'sum')
    )
    .reset_index()
)

print("Valenbisi por barrio (top 10 por capacidad):")
display(df_valen_barrio.sort_values('capacidad_total', ascending=False).head(10))

Valenbisi por barrio (top 10 por capacidad):


,nombre,num_estaciones,capacidad_total
31,la carrasca,12,321
64,sant pau,11,203
3,benicalap,11,199
8,cabanyal-canyamelar,11,198
2,arrancapins,9,190
13,ciutat de les arts i de les ciencies,7,160
47,malilla,7,155
54,nou moles,8,146
60,sant francesc,6,146
12,campanar,7,145


**CÁLCULO DEL Mobility_Score**

 Si tenemos capacidad, la usamos como métrica principal
 (refleja mejor la oferta real del servicio).

Si no, usamos el número de estaciones.


In [79]:
col_metrica = 'capacidad_total' if 'capacidad_total' in df_valen_barrio.columns else 'num_estaciones'

scaler_v = MinMaxScaler()
df_valen_barrio['Mobility_Score'] = (
    scaler_v.fit_transform(df_valen_barrio[[col_metrica]])
) * 100

df_valen_final = df_valen_barrio[['nombre', 'Mobility_Score']].rename(columns={'nombre': 'barrio'})

# Barrios sin ninguna estación reciben score 0
df_valen_final['Mobility_Score'] = df_valen_final['Mobility_Score'].fillna(0)

print("Mobility_Score por barrio (top 10):")
display(df_valen_final.sort_values('Mobility_Score', ascending=False).head(10))

Mobility_Score por barrio (top 10):


,barrio,Mobility_Score
31,la carrasca,100.000000
64,sant pau,61.437908
3,benicalap,60.130719
8,cabanyal-canyamelar,59.803922
2,arrancapins,57.189542
13,ciutat de les arts i de les ciencies,47.385621
47,malilla,45.751634
54,nou moles,42.810458
60,sant francesc,42.810458
12,campanar,42.483660


In [80]:
print(f"Barrios con dato: {len(df_valen_final)}")

Barrios con dato: 71


**TRATAMIENTO DE BARRIOS SIN ESTACIONES VALENBISI**

17 barrios no tienen ninguna estación Valenbisi dentro de sus límites. En su mayoría son pedanías rurales o barrios periféricos donde el servicio no opera.

Les asignamos score 0 porque la ausencia de estaciones es un dato real, no una limitación de cobertura.

In [81]:
todos_los_barrios = pd.DataFrame({'barrio': gdf_barrios['nombre']})
df_valen_final = todos_los_barrios.merge(df_valen_final, on='barrio', how='left')

print("Barrios sin estaciones Valenbisi:")
print(df_valen_final[df_valen_final['Mobility_Score'].isna()]['barrio'].tolist())

df_valen_final['Mobility_Score'] = df_valen_final['Mobility_Score'].fillna(0)

print(f"\nTotal barrios        : {len(df_valen_final)}")
print(f"Con estaciones       : {df_valen_final['Mobility_Score'].gt(0).sum()}")
print(f"Sin estaciones       : {df_valen_final['Mobility_Score'].eq(0).sum()}")

Barrios sin estaciones Valenbisi:
['rafalell-vistabella', 'el palmar', 'el saler', 'el perellonet', 'carpesa', 'borboto', 'benifaraig', 'el pilar', 'poble nou', 'les cases de barcena', 'mahuella-tauladella', 'pinedo', 'massarrojos', 'faitanar', "castellar-l'oliveral", "el forn d'alcedo", 'la torre']

Total barrios        : 88
Con estaciones       : 69
Sin estaciones       : 19


---
## 6. Tabla maestra y cálculo del UQI

Unimos todos los scores a nivel de barrio y calculamos el **Urban Quality Index (UQI)**,
un indicador compuesto que resume la calidad urbana en una única puntuación (0–100).

In [92]:
# Comprobación rápida de que tenemos los 5 dataframes listos
print(f"df_aire_final   : {len(df_aire_final)} barrios")
print(f"df_quejas_final : {len(df_quejas_final)} barrios")
print(f"df_ruido_final  : {len(df_ruido_final)} barrios")
print(f"df_equip_final  : {len(df_equip_final)} barrios")
print(f"df_valen_final  : {len(df_valen_final)} barrios")

df_aire_final   : 88 barrios
df_quejas_final : 88 barrios
df_ruido_final  : 88 barrios
df_equip_final  : 88 barrios
df_valen_final  : 88 barrios


**CONSTRUCCIÓN DE LA TABLA MAESTRA**

Partimos de todos los barrios del GeoJSON y añadimos
 cada score mediante left join sobre la columna 'barrio'.

In [93]:
df_master = gdf_barrios[['nombre']].rename(columns={'nombre': 'barrio'}).copy()

df_master = df_master.merge(df_aire_final,    on='barrio', how='left')
df_master = df_master.merge(
    df_quejas_final[['barrio_localización', 'Complaints_Score']]
    .rename(columns={'barrio_localización': 'barrio'}),
    on='barrio', how='left'
)
df_master = df_master.merge(df_ruido_final[['barrio', 'Noise_Score']],   on='barrio', how='left')
df_master = df_master.merge(df_equip_final,  on='barrio', how='left')
df_master = df_master.merge(df_valen_final,  on='barrio', how='left')

print(f"Tabla maestra: {df_master.shape[0]} barrios × {df_master.shape[1]} columnas")
print("\nNulos por columna:")
display(df_master.isnull().sum())

Tabla maestra: 88 barrios × 6 columnas

Nulos por columna:


barrio               0
Air_Quality_Score    0
Complaints_Score     0
Noise_Score          0
Services_Score       0
Mobility_Score       0
dtype: int64

**IMPUTACIÓN DE VALORES FALTANTES**

Usamos la mediana (más robusta que la media ante outliers)
 para los barrios sin datos en alguna de las fuentes.


In [94]:
score_cols = [
    'Air_Quality_Score',
    'Complaints_Score',
    'Noise_Score',
    'Services_Score',
    'Mobility_Score'
]

for col in score_cols:
    median_val = df_master[col].median()
    df_master[col] = df_master[col].fillna(median_val)

print("Nulos tras imputación:", df_master[score_cols].isnull().sum().sum())

Nulos tras imputación: 0


**CÁLCULO DEL URBAN QUALITY INDEX (UQI)**

Promedio ponderado de los cinco scores.

Los pesos reflejan la importancia relativa de cada dimensión en la calidad de vida urbana:

- Air_Quality_Score  → 0.25  (salud ambiental)
- Complaints_Score   → 0.20  (percepción ciudadana)
- Noise_Score        → 0.20  (confort acústico)
- Services_Score     → 0.20  (dotación de servicios)
- Mobility_Score     → 0.15  (movilidad sostenible)

Los pesos suman 1.0. Pueden ajustarse según criterio experto.

In [95]:
pesos = {
    'Air_Quality_Score': 0.25,
    'Complaints_Score':  0.20,
    'Noise_Score':       0.20,
    'Services_Score':    0.20,
    'Mobility_Score':    0.15
}

df_master['UQI'] = sum(
    df_master[col] * peso
    for col, peso in pesos.items()
)

df_master['UQI'] = df_master['UQI'].round(2)

print("Top 10 barrios por UQI:")
display(
    df_master[['barrio', 'UQI'] + score_cols]
    .sort_values('UQI', ascending=False)
    .reset_index(drop=True)
    .head(10)
)

Top 10 barrios por UQI:


,barrio,UQI,Air_Quality_Score,Complaints_Score,Noise_Score,Services_Score,Mobility_Score
0,la carrasca,66.99,59.256524,87.268128,28.235294,70.370370,100.000000
1,cabanyal-canyamelar,60.48,59.256524,44.603710,50.000000,88.888889,59.803922
2,la xerea,60.44,46.828851,76.222597,82.352941,74.074074,14.705882
3,sant francesc,54.26,31.406552,49.915683,50.000000,100.000000,42.810458
4,mestalla,53.26,46.828851,46.711636,97.058824,37.037037,35.947712
5,exposicio,53.22,46.828851,71.332209,50.000000,59.259259,35.947712
6,ciutat universitaria,52.50,46.828851,90.978078,50.000000,33.333333,39.542484
7,el carme,52.41,31.406552,68.634064,50.000000,83.333333,27.777778
8,el pla del remei,51.86,31.406552,75.716695,73.529412,42.592593,37.581699
9,el saler,51.01,50.771543,91.231029,94.823529,5.555556,0.000000


In [96]:
# exportación de la tabla maestra

# csv
df_master.to_csv('uqi_barrios.csv', index=False)
print(" uqi_barrios.csv guardado")

# GeoJSON: merge con geometría original de barrios
gdf_uqi = gdf_barrios[['nombre', 'geometry']].rename(columns={'nombre': 'barrio'})
gdf_uqi = gdf_uqi.merge(df_master, on='barrio', how='left')
gdf_uqi.to_file('uqi_barrios.geojson', driver='GeoJSON')
print(" uqi_barrios.geojson guardado")

print(f"\nResumen final:")
print(f"  Barrios procesados : {len(df_master)}")
print(f"  UQI medio          : {df_master['UQI'].mean():.1f}")
print(f"  UQI máximo         : {df_master['UQI'].max():.1f} → {df_master.loc[df_master['UQI'].idxmax(), 'barrio']}")
print(f"  UQI mínimo         : {df_master['UQI'].min():.1f} → {df_master.loc[df_master['UQI'].idxmin(), 'barrio']}")

 uqi_barrios.csv guardado
 uqi_barrios.geojson guardado

Resumen final:
  Barrios procesados : 88
  UQI medio          : 43.8
  UQI máximo         : 67.0 → la carrasca
  UQI mínimo         : 31.0 → sant marcel.li
